# TAFIP Framework: Machine Learning & NLP Sentiment Training Engine

**Project Title**: Investigating the Role of Trust and Security in Fintech Product Adoption and Developing a Trust Assessment Dashboard: A Comparative Study of Parkway (Nigeria) and Revolut (UK).

This notebook performs:
1. **Data Ingestion & Preprocessing**: Clean `tafip.csv`, extract 100 responses (50 Parkway, 50 Revolut), Likert mapping, reverse scoring for risk items.
2. **NLP Sentiment & Frustration Engine**: Parse 6 interview transcripts (`.docx`) for VADER sentiment scores and Frustration metrics.
3. **Predictive Model Training**: Benchmark **Random Forest Regressor** vs **XGBoost Regressor** predicting consolidated Trust Score (0-100).
4. **Export Artifacts**: Save winning `.pkl` model, metadata JSON, and processed datasets.

In [6]:
!python3 --version

Python 3.9.6


In [2]:
import sys, os, json, joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate

sys.path.append('..')
from backend.app.services.preprocessing import clean_and_preprocess_df
from backend.app.services.nlp_engine import process_all_interviews

ModuleNotFoundError: No module named 'nltk'

## 1. Load Survey Dataset and Execute 50/50 Partition

In [ ]:
csv_path = '../tafip.csv'
raw_df = pd.read_csv(csv_path, encoding='utf-8-sig')
df_100, X = clean_and_preprocess_df(raw_df)
y = df_100['Trust_Score_Target']

print(f'Ingested shape: {df_100.shape}')
print(f'Parkway count: {sum(df_100["Target_App"] == "Parkway")}')
print(f'Revolut count: {sum(df_100["Target_App"] == "Revolut")}')
df_100[['Target_App', 'Trust_Perception_Score', 'Perceived_Security_Score', 'Trust_Score_Target']].head()

## 2. Qualitative Interview NLP Sentiment Engine

In [ ]:
nlp_results = process_all_interviews('..')
print('NLP Sentiment Summary across 6 Interview Transcripts:')
print(json.dumps(nlp_results['summary'], indent=2))

## 3. Algorithm Training & Cross-Validation Benchmark (Random Forest vs XGBoost)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)

rf_scores = cross_validate(rf_model, X, y, cv=kf, scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'])
xgb_scores = cross_validate(xgb_model, X, y, cv=kf, scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'])

print('Random Forest Scores:')
print(f'  R2:   {np.mean(rf_scores["test_r2"]):.4f}')
print(f'  MAE:  {-np.mean(rf_scores["test_neg_mean_absolute_error"]):.4f}')
print(f'  RMSE: {-np.mean(rf_scores["test_neg_root_mean_squared_error"]):.4f}')

print('\nXGBoost Scores:')
print(f'  R2:   {np.mean(xgb_scores["test_r2"]):.4f}')
print(f'  MAE:  {-np.mean(xgb_scores["test_neg_mean_absolute_error"]):.4f}')
print(f'  RMSE: {-np.mean(xgb_scores["test_neg_root_mean_squared_error"]):.4f}')

## 4. Model Selection & Export Artifacts

In [ ]:
best_model = rf_model if np.mean(rf_scores['test_r2']) >= np.mean(xgb_scores['test_r2']) else xgb_model
best_model.fit(X, y)

os.makedirs('../backend/app/models', exist_ok=True)
joblib.dump(best_model, '../backend/app/models/tafip_trust_model.pkl')
print('Model exported successfully!')